In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGE_INDEX_API_KEY")
CHAT_API_KEY = os.getenv("CHAT_API_KEY")

print("keys loaded successfully")

keys loaded successfully


In [2]:
from pageindex import PageIndexClient
from openai import OpenAI

page_index_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=CHAT_API_KEY, base_url=os.getenv("CHAT_API_BASE_URL"))

print("PageIndexClient initialized successfully")

PageIndexClient initialized successfully


In [3]:
PDF_PATH = "Information_Retrieval_and_Text_Processing_Guide_6308.pdf"

result = page_index_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(doc_id)

pi-cmw071wkr005n0annqgd1rfq1


/tmp/ipykernel_1627865/3991522647.py:3: UserWarning: Document "Information_Retrieval_and_Text_Processing_Guide_6308.pdf" was stored as "Information_Retrieval_and_Text_Processing_Guide_6308_2.pdf".
  result = page_index_client.submit_document(PDF_PATH)


In [4]:
# pageIndex take some time to build the tree index
while True:
    status_result = page_index_client.get_document(doc_id)
    status = status_result.get("status", {})
    if status == "completed":
        print("Document is ready for querying.")
        break
    else:
        print("Document is still being processed. Waiting for 5 seconds...")
        break
        time.sleep(5)

Document is ready for querying.


In [5]:
# Inspect the tree structure of the document
tree_result = page_index_client.get_tree(doc_id, node_summary=True)
page_index_tree = tree_result.get("result", {})

print(f" Top-level sections: {len(page_index_tree)}")
print("\n Raw tree (first node):")
print(json.dumps(page_index_tree[0] if page_index_tree else {}, indent=2))

 Top-level sections: 1

 Raw tree (first node):
{
  "title": "Information Retrieval & Text Processing",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 15,
  "summary": "This section provides a comprehensive guide to modern text chunking strategies for Information Retrieval and RAG systems. It covers the evolution of search from lexical (TF-IDF, BM25) to semantic vector search, fundamentals of document parsing (including OCR and structural extraction), and complex structural elements in nested hierarchies. It details various chunking strategies\u2014comparing fixed-size, recursive, document-based, and semantic approaches\u2014along with the mechanics of overlap, sliding windows, and semantic boundary detection. Furthermore, it addresses vector embeddings and chunk size trade-offs, outlines evaluation metrics for both retrieval and generation, and concludes with future directions like layout-aware and multi-modal parsing.",
  "text": "",
  "nodes": [
    {
      "title": "Inform

In [8]:
print(json.dumps(page_index_tree[0]["nodes"][1] if page_index_tree else {}, indent=2))

{
  "title": "Abstract",
  "node_id": "0002",
  "start_index": 1,
  "end_index": 3,
  "summary": "The text introduces a standardized testbed document designed for evaluating NLP text chunking strategies in RAG systems and provides a comprehensive 13-chapter table of contents covering information retrieval, document parsing, chunking algorithms, and evaluation metrics.",
  "text": "## Abstract\n\nThis document serves as a standardized testbed for evaluating Natural Language Processing (NLP) text chunking strategies. It deliberately incorporates a wide array of typographical and structural elements\u2014ranging from dense paragraphs and deep hierarchical headings to tabular data and bulleted lists. The purpose of this guide is to provide a robust corpus for algorithms designed to slice documents into semantically coherent vectors for Retrieval-Augmented Generation (RAG) systems.\n\nIR & Text Processing\n\nPage 1 of 15\n\nTable of Contents\n\n1. Introduction to Information Retrieval Page 

In [9]:
# --------- preety-print the full tree structure of the document
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = " " * indent + ("--" if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix} [{node['node_id']}] {node.get('title', 'No Title')} (Page: {page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)
    
print_tree(page_index_tree)

 [0000] Information Retrieval & Text Processing (Page: ?)
 -- [0001] Information Retrieval & Text Processing (intro) (Page: ?)
 -- [0002] Abstract (Page: ?)
 -- [0003] 1. Introduction to Information Retrieval (Page: ?)
 -- [0004] 2. The Evolution of Search Systems (Page: ?)
 -- [0005] 3. Fundamentals of Document Parsing (Page: ?)
 -- [0006] 4. Structural Elements in Text (Page: ?)
 -- [0007] 5. Introduction to Text Chunking (Page: ?)
  -- [0008] 5. Introduction to Text Chunking (intro) (Page: ?)
  -- [0009] 5.1 Comparison of Common Strategies (Page: ?)
 -- [0010] 7. Overlap and Sliding Windows (Page: ?)
 -- [0011] 8. Semantic Boundary Detection (Page: ?)
  -- [0012] 8. Semantic Boundary Detection (intro) (Page: ?)
  -- [0013] 8.1 Sentence Splitting (Page: ?)
  -- [0014] 8.2 Paragraph Splitting (Page: ?)
   -- [0015] 8.2 Paragraph Splitting (intro) (Page: ?)
   -- [0016] Advanced NLP Segmentation (Page: ?)
 -- [0017] 11. Vector Embeddings and Chunk Size (Page: ?)
 -- [0018] 12. Evaluati

In [10]:
def count_nodes(nodes):
    count = 0
    for node in nodes:
        count += 1
        if node.get("nodes"):
            count += count_nodes(node["nodes"])
    return count

node_count = count_nodes(page_index_tree)
print(f"Total nodes in the tree: {node_count}")

Total nodes in the tree: 20


In [15]:
# Vectorless RAG (Retrieval-Augmented Generation) with PageIndex

def llm_tree_search(query, tree)->dict:
    """_summary_

    Args:
        query (_type_): _description_
        tree (_type_): _description_

    Returns:
        dict: _description_
    """
    # compress tree to only include node_id, title, page_index, and summary for saving tokens
    def compress(nodes):
        out = []
        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node.get("title", ""),
                "page": node.get("page_index", "?"),
                "summary": node.get("text", "")[:150],
            }
            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a tree structure of a document and a query. Your task is to find the most relevant node in the tree that answers the query. The tree is represented as a list of nodes, where each node has a 'node_id', 'title', 'page', 'summary', and optionally 'children' which is a list of child nodes.Think step by step and return the node_id of the most relevant node. If no relevant node is found, return 'None'.
    Query: {query}
    Document Tree: {json.dumps(compressed_tree, indent=2)}
    
    Reply ONLY in this exact JSON format:
    {{
        "thinking": "<your step-by-step reasoning here>",
        "node_list": ["<list of node_ids you considered>"],
    }}
    """
    
    model = os.getenv("CHAT_MODEL", "minimax-local")
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that finds relevant nodes in a document tree based on a query."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        response_format={"type": "json_object"}
    )   
    
    return response
    

In [23]:
# --- test with a sample query
query = "What is the definition of Information Retrieval?"
result = llm_tree_search(query, page_index_tree)

In [24]:
response_content = result.choices[0].message.content
response_json = json.loads(response_content)
print(json.dumps(response_json, indent=2))

{
  "thinking": "The query asks for the definition of Information Retrieval. The document tree contains a node titled '1. Introduction to Information Retrieval' (node_id 0003) whose summary begins with 'Information Retrieval (IR) is the science of searching for information in a document, searching for docum...' which directly provides the definition. Other nodes are less relevant as they focus on chunking, parsing, or other topics.",
  "node_list": [
    "0003"
  ]
}


In [25]:
# --- test with a sample query
query2 = "Tell me about Semantic Boundary Detection?"
result2 = llm_tree_search(query2, page_index_tree)

In [26]:
response_content2 = result2.choices[0].message.content
response_json2 = json.loads(response_content2)
print(json.dumps(response_json2, indent=2))

{
  "thinking": "The query asks about 'Semantic Boundary Detection'. The document tree has a node with title '8. Semantic Boundary Detection' (node_id 0011) which has children including an intro (0012), '8.1 Sentence Splitting' (0013), and '8.2 Paragraph Splitting' (0014) with a child 'Advanced NLP Segmentation' (0016). The most relevant node is the main section node 0011, as it directly addresses the topic.",
  "node_list": [
    "0011",
    "0012",
    "0013",
    "0014",
    "0016"
  ]
}


In [29]:
def find_nodes_by_ids(tree: list, target_ids:list)->list:
    """Find nodes in the tree by their node_ids.

    Args:
        tree (list): The document tree.
        target_ids (list): List of node_ids to find.

    Returns:
        list: List of nodes that match the target_ids.
    """
    found_nodes = []

    def search(nodes):
        for node in nodes:
            if node["node_id"] in target_ids:
                found_nodes.append(node)
            if node.get("nodes"):
                found_nodes.extend(find_nodes_by_ids(node["nodes"], target_ids))

    search(tree)
    return found_nodes

In [50]:
# -----  Generate answer from retrieved nodes ------------------

def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."

    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"  # <-- Fixed here
        )
    
    context = "\n\n---\n\n".join(context_parts)
    # ... rest of your code to generate the answer using the model
    # Construct the grounded system prompt
    prompt = f"""You are an expert document analyst.
    Answer the question using ONLY the provided context.
    For every claim you make, cite the section title and page number in parentheses.
    Be concise and precise.

    Question: {query}

    Context:
    {context}

    Answer:"""

    # Generate response from OpenAI
    model = os.getenv("CHAT_MODEL", "minimax-local")
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )   

    # Return the extracted string text response
    return response.choices[0].message.content

In [51]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  -> finds relevant node_ids
    Step 2: Node Retrieval   -> fetches section content
    Step 3: Answer Generation -> produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids      = json.loads(search_result.choices[0].message.content)["node_list"]
    if verbose:
        print(f" Found {len(node_ids)} relevant node ID(s) via tree search.")

    # Step 2: Node Retrieval
    # Assumes a retrieval helper function exists to match IDs to nodes in your tree structure
    nodes = find_nodes_by_ids(tree, node_ids) 
    
    if verbose:
        print(f" Reconstructed {len(nodes)} document sections for context.")

    # Step 3: Answer Generation
    if verbose:
        print(" Generating grounded answer...")
        
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"{'='*55}\nDone!\n{'='*55}")
        
    return answer

In [52]:
answer = vectorless_rag("What is the definition of Information Retrieval?", page_index_tree)

🔍 Query: What is the definition of Information Retrieval?
 Found 1 relevant node ID(s) via tree search.
 Reconstructed 1 document sections for context.
 Generating grounded answer...
Done!


In [53]:
answer

'Based on the provided context, Information Retrieval (IR) is "the science of searching for information in a document, searching for documents themselves, and also searching for metadata that describes data, as well as searching within databases and the World Wide Web" (Section: \'1. Introduction to Information Retrieval\' | Page 3 of 15).'

In [54]:
#--- test a few different queries

test_queries = [
    "Tell about semantic boundary detection.",
    "what is vector embedding and how does chunk size impact it ?",
    "what is text chunking and how does it impact vector embedding?"
]

for q in test_queries:
    print(f"\n\n{'='*55}\nQuery: {q}\n{'='*55}")
    ans = vectorless_rag(q, page_index_tree)
    print(f"Answer:\n{ans}")



Query: Tell about semantic boundary detection.
🔍 Query: Tell about semantic boundary detection.
 Found 5 relevant node ID(s) via tree search.
 Reconstructed 5 document sections for context.
 Generating grounded answer...
Done!
Answer:
Semantic boundary detection is a state-of-the-art method in text chunking that analyzes text to identify natural transition points where one topic ends and another begins, rather than relying on arbitrary lengths (Section: 8. Semantic Boundary Detection). The most basic form is sentence splitting, where text is divided into sentences using NLP libraries like spaCy or NLTK, and a chunker aggregates sentences until approaching the token limit, ensuring cuts only happen at punctuation marks so sentences are never torn apart (Section: 8.1 Sentence Splitting).


Query: what is vector embedding and how does chunk size impact it ?
🔍 Query: what is vector embedding and how does chunk size impact it ?
 Found 1 relevant node ID(s) via tree search.
 Reconstructed 